# Strawberry Yield Prediction — Full Pipeline

**Two-stage system:**
- Stage 1: LightGBM yield prediction per grid cell
- Stage 2: Growth-rate rules → optimal harvest timing

## Section 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, subprocess, importlib

# ── Configuration — update these paths to match your Google Drive layout ──
GITHUB_REPO  = 'https://github.com/ChristinaPuth/strawberry-yield.git'
REPO_LOCAL   = '/content/strawberry-yield'
BASE_DATA    = ('/content/drive/MyDrive/YOUR_PROJECT_FOLDER/YieldMapHarvest_Original Data')
OUTPUTS      = ('/content/drive/MyDrive/YOUR_PROJECT_FOLDER/outputs/processed_data')

if not os.path.exists(REPO_LOCAL):
    print('First clone...')
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_LOCAL], check=True)
else:
    print('Pulling latest...')
    result = subprocess.run(['git','-C',REPO_LOCAL,'pull'], capture_output=True, text=True)
    print(result.stdout.strip())

SRC = REPO_LOCAL + '/src'
if SRC not in sys.path: sys.path.insert(0, SRC)

for mod in ['data_pipeline','feature_engineering','models','harvest_advisor','visualize']:
    sys.modules.pop(mod, None)

import data_pipeline
import feature_engineering as fe
import models as m
import harvest_advisor as ha
import visualize as viz
import pandas as pd, numpy as np, matplotlib.pyplot as plt

print('All modules loaded from GitHub')
print(f'   days_since_last NOT in ALL_FEATS: {"days_since_last" not in fe.ALL_FEATS}')
print(f'   seasonal_mean in fe             : {"seasonal_mean" in open(fe.__file__).read()}')
print(f'   derive_thresholds in ha         : {"derive_thresholds" in open(ha.__file__).read()}')


In [ ]:
!pip install openmeteo-requests requests-cache retry-requests lightgbm xgboost -q

In [ ]:
import subprocess, sys
result = subprocess.run(['git','-C',REPO_LOCAL,'pull'], capture_output=True, text=True)
print(result.stdout.strip() or 'Already up to date.')
for mod in ['data_pipeline','feature_engineering','models',
            'harvest_advisor','visualize','validation_schemes']:
    sys.modules.pop(mod, None)
import data_pipeline
import feature_engineering as fe
import models as m
import harvest_advisor as ha
import visualize as viz
import validation_schemes as vs
print('All modules reloaded')


## Section 1 — Data Loading

### Cell 3 — Load raw harvest data

In [ ]:
df_sm  = data_pipeline.load_site('SantaMaria', BASE_DATA)
df_sal = data_pipeline.load_site('Salinas',    BASE_DATA)

print('\nSantaMaria summary:')
print(data_pipeline.summary(df_sm).to_string(index=False))
print('\nSalinas summary:')
print(data_pipeline.summary(df_sal).to_string(index=False))


### Cell 4 — Load weather data (cached)

In [ ]:
weather_sm  = data_pipeline.load_weather_cached('SantaMaria', OUTPUTS)
weather_sal = data_pipeline.load_weather_cached('Salinas',    OUTPUTS)
print(f'SantaMaria: {len(weather_sm)} days  |  Salinas: {len(weather_sal)} days')


## Section 2 — Baseline Model (1×1 grid)

Build features on the original grid resolution, run ablation, compare models, evaluate on test set.

### Cell 5 — Build features (n=1)

In [ ]:
# Set force_rebuild=True the first time, then switch back to False
df_feat_sm  = data_pipeline.load_features_cached('SantaMaria', df_sm,  weather_sm,  OUTPUTS, force_rebuild=False)
df_feat_sal = data_pipeline.load_features_cached('Salinas',    df_sal, weather_sal, OUTPUTS, force_rebuild=False)


### Cell 6 — Split data

SantaMaria: train 1–20, val 21–23, test 24–27. Salinas: train 1–15, val 16–18, test 19–20.

In [ ]:
splits_sm  = fe.split_data(df_feat_sm,  'SantaMaria')
splits_sal = fe.split_data(df_feat_sal, 'Salinas')

# Fix rolling_mean_3 NaN at harvest_idx=2
for splits in [splits_sm, splits_sal]:
    for k in splits:
        mask = splits[k]['rolling_mean_3'].isna()
        if mask.sum() > 0:
            splits[k].loc[mask, 'rolling_mean_3'] = splits[k].loc[mask, 'yield_lag1']
            print(f'  Fixed {mask.sum():,} NaN rows (harvest_idx={splits[k].loc[mask,"harvest_idx"].unique()})')

for name, splits in [('SantaMaria', splits_sm), ('Salinas', splits_sal)]:
    total_nan = sum(splits[k]['rolling_mean_3'].isna().sum() for k in splits)
    print(f'  {name}: rolling_mean_3 NaN after fix = {total_nan}')

### Cell 7 — Derive Stage 2 thresholds

In [ ]:
thresholds_sm  = ha.derive_thresholds(splits_sm['train'],  site='SantaMaria')
thresholds_sal = ha.derive_thresholds(splits_sal['train'], site='Salinas')

### Cell 8 — Ablation study

In [ ]:
ablation_sm  = m.run_ablation(splits_sm,  'SantaMaria')
ablation_sal = m.run_ablation(splits_sal, 'Salinas')

print('\nSantaMaria ablation:')
print(ablation_sm[['config','n_features','val_r2','val_rmse','val_mae','train_r2','description']].to_string(index=False))
print('\nSalinas ablation:')
print(ablation_sal[['config','n_features','val_r2','val_rmse','val_mae','train_r2','description']].to_string(index=False))

### Cell 9 — Ablation visualisation

In [ ]:
viz.plot_ablation_table(ablation_sm, ablation_sal)

### Cell 10 — Model comparison (Stage 1)

In [ ]:
# SantaMaria 最优特征集
features_sm  = m.ABLATION_CONFIGS['A3']
# Salinas 最优特征集
features_sal = m.ABLATION_CONFIGS['A4']

# ── NaN 修复（Salinas rolling_mean_3 在第一次收获时为 NaN）──────────────────
splits_sal_clean = {k: v.copy() for k, v in splits_sal.items()}
for k in splits_sal_clean:
    mask = splits_sal_clean[k]['rolling_mean_3'].isna()
    splits_sal_clean[k].loc[mask, 'rolling_mean_3'] = \
        splits_sal_clean[k].loc[mask, 'yield_lag1']

model_results_sm = m.run_model_comparison(
    splits_sm, 'SantaMaria', features_sm)
model_results_sal = m.run_model_comparison(
    splits_sal_clean, 'Salinas', features_sal)   # ← 用 clean 版

print('\nSantaMaria — model comparison (val):')
print(model_results_sm[['model','val_r2','val_rmse','train_r2']].to_string(index=False))
print('\nSalinas — model comparison (val):')
print(model_results_sal[['model','val_r2','val_rmse','train_r2']].to_string(index=False))

### Cell 11 — Feature importance

In [ ]:
m.plot_feature_importance(model_results_sm)
m.plot_feature_importance(model_results_sal)

### Cell 12 — Ground truth vs predicted line chart

In [ ]:
viz.plot_ground_truth_line(df_feat_sm,  model_results_sm,  'SantaMaria')
viz.plot_ground_truth_line(df_feat_sal, model_results_sal, 'Salinas')

### Cell 13 — Scatter plot (cell level)

In [ ]:
viz.plot_prediction_scatter(df_feat_sm,  model_results_sm,  'SantaMaria')
viz.plot_prediction_scatter(df_feat_sal, model_results_sal, 'Salinas')

### Cell 14 — Yield map (Ground Truth | Predicted | Error)

In [ ]:
print('SantaMaria test dates:', sorted(splits_sm['test']['harvest_date'].unique()))
print('Salinas test dates:',    sorted(splits_sal['test']['harvest_date'].unique()))

# Plot all test dates
for date in sorted(splits_sm['test']['harvest_date'].unique()):
    viz.plot_ground_truth_map(df_feat_sm, model_results_sm, date, 'SantaMaria')
for date in sorted(splits_sal['test']['harvest_date'].unique()):
    viz.plot_ground_truth_map(df_feat_sal, model_results_sal, date, 'Salinas')

### Cell 15 — Test set evaluation

⚠️ Run once only after all design decisions are final.

In [ ]:
test_results_sm  = m.evaluate_on_test(model_results_sm,  splits_sm,  'SantaMaria')
test_results_sal = m.evaluate_on_test(model_results_sal, splits_sal, 'Salinas')

rows = []
for site, res in [('SantaMaria', test_results_sm), ('Salinas', test_results_sal)]:
    rows.append({
        'Site': site, 'Model': res['model'],
        'Test R²': res['test_r2'], 'Test RMSE': res['test_rmse'], 'Test MAE': res['test_mae'],
    })
summary_df = pd.DataFrame(rows)
print('\n' + '='*60)
print('  FINAL TEST SET ACCURACY SUMMARY (Stage 1, n=1)')
print('='*60)
print(summary_df.to_string(index=False))
print('='*60)

### Cell 16 — Harvest advisor (Stage 1 + Stage 2)

In [ ]:
advice_sm = ha.recommend_harvest(
    model_results     = model_results_sm,
    df_raw            = df_sm,
    weather           = weather_sm,
    site              = 'SantaMaria',
    last_harvest_date = pd.Timestamp('2024-07-09'),
    thresholds        = thresholds_sm,
)
ha.print_recommendation(advice_sm)
ha.plot_advice(advice_sm)

advice_sal = ha.recommend_harvest(
    model_results     = model_results_sal,
    df_raw            = df_sal,
    weather           = weather_sal,
    site              = 'Salinas',
    last_harvest_date = pd.Timestamp('2024-08-10'),
    thresholds        = thresholds_sal,
)
ha.print_recommendation(advice_sal)
ha.plot_advice(advice_sal)

### Cell 17 — Decision Quality

In [ ]:
dq_sm  = ha.decision_quality(advice_sm,  df_raw=df_sm,  splits=splits_sm)
dq_sal = ha.decision_quality(advice_sal, df_raw=df_sal, splits=splits_sal)
viz.plot_dq_summary(dq_sm, dq_sal)

## Section 2.5 — Grid Shape Scan (Original Train/Val/Test Split)

Scan grid shapes using the original chronological split
(SantaMaria: train 1–20, val 21–23, test 24–27).
Compare best ablation config and val R² across shapes.

In [ ]:
import importlib
importlib.reload(viz)

# Ground truth vs predicted for each grid shape
grid_viz_configs = [
    ('1x1', 'SantaMaria', df_feat_sm,  model_results_sm),
    ('1x1', 'Salinas',    df_feat_sal, model_results_sal),
]

for label, nx, ny, desc in configs:
    if nx == 1 and ny == 1:
        continue

    feat_sm  = fe.build_features(
        fe.coarsen_grid(df_sm,  nx=nx, ny=ny),
        'SantaMaria', weather_sm, backfill_lags=True, drop_anomaly=True)
    feat_sal = fe.build_features(
        fe.coarsen_grid(df_sal, nx=nx, ny=ny),
        'Salinas', weather_sal, backfill_lags=True, drop_anomaly=True)

    sp_sm  = fe.split_data(feat_sm,  'SantaMaria')
    sp_sal = fe.split_data(feat_sal, 'Salinas')

    # Fix rolling_mean_3 NaN
    for splits in [sp_sm, sp_sal]:
        for k in splits:
            mask = splits[k]['rolling_mean_3'].isna()
            if mask.sum() > 0:
                splits[k].loc[mask, 'rolling_mean_3'] = splits[k].loc[mask, 'yield_lag1']

    abl_sm  = m.run_ablation(sp_sm,  'SantaMaria')
    abl_sal = m.run_ablation(sp_sal, 'Salinas')

    best_cfg_sm  = abl_sm[~abl_sm['is_baseline']].iloc[0]['config']
    best_cfg_sal = abl_sal[~abl_sal['is_baseline']].iloc[0]['config']

    feats_sm  = m.ABLATION_CONFIGS[best_cfg_sm]
    feats_sal = m.ABLATION_CONFIGS[best_cfg_sal]

    # Fix NaN in feature columns before model comparison (for LinearRegression)
    sp_sm_clean  = {k: v.copy() for k, v in sp_sm.items()}
    sp_sal_clean = {k: v.copy() for k, v in sp_sal.items()}

    for sp_clean, feats in [(sp_sm_clean, feats_sm), (sp_sal_clean, feats_sal)]:
        for k in sp_clean:
            for col in feats:
                if col in sp_clean[k].columns:
                    nan_mask = sp_clean[k][col].isna()
                    if nan_mask.sum() > 0:
                        sp_clean[k].loc[nan_mask, col] = sp_clean[k][col].median()

    mr_sm  = m.run_model_comparison(sp_sm_clean,  'SantaMaria', feats_sm)
    mr_sal = m.run_model_comparison(sp_sal_clean, 'Salinas',    feats_sal)

    grid_viz_configs.append((label, 'SantaMaria', feat_sm,  mr_sm))
    grid_viz_configs.append((label, 'Salinas',    feat_sal, mr_sal))

# Plot ground truth vs predicted line charts
for label, site, df_feat, model_res in grid_viz_configs:
    print(f"\n  {label} — {site}")
    viz.plot_ground_truth_line(df_feat, model_res, f'{site} ({label})')

## Section 3 — Grid Shape × Validation Scheme Cross Experiment

Build features for multiple grid shapes (1×1 to 8×8), run A.1/B/C/D/E schemes, compare results.

### Cell 18 — Build all grid shape features

In [ ]:
import importlib
importlib.reload(fe); importlib.reload(vs)

# 1x1: load from cache
df_feat_sm  = data_pipeline.load_features_cached('SantaMaria', df_sm,  weather_sm,  OUTPUTS)
df_feat_sal = data_pipeline.load_features_cached('Salinas',    df_sal, weather_sal, OUTPUTS)

# 5x5
df_feat_sm_5x5  = fe.build_features(fe.coarsen_grid(df_sm,  nx=5, ny=5), 'SantaMaria', weather_sm,  backfill_lags=True, drop_anomaly=True)
df_feat_sal_5x5 = fe.build_features(fe.coarsen_grid(df_sal, nx=5, ny=5), 'Salinas',    weather_sal, backfill_lags=True, drop_anomaly=True)
print('5x5 done.')

# 7x7
df_feat_sm_7x7  = fe.build_features(fe.coarsen_grid(df_sm,  nx=7, ny=7), 'SantaMaria', weather_sm,  backfill_lags=True, drop_anomaly=True)
df_feat_sal_7x7 = fe.build_features(fe.coarsen_grid(df_sal, nx=7, ny=7), 'Salinas',    weather_sal, backfill_lags=True, drop_anomaly=True)
print('7x7 done.')

# 8x8
df_feat_sm_8x8  = fe.build_features(fe.coarsen_grid(df_sm,  nx=8, ny=8), 'SantaMaria', weather_sm,  backfill_lags=True, drop_anomaly=True)
df_feat_sal_8x8 = fe.build_features(fe.coarsen_grid(df_sal, nx=8, ny=8), 'Salinas',    weather_sal, backfill_lags=True, drop_anomaly=True)
print('8x8 done.')

# 2x8
df_feat_sm_2x8  = fe.build_features(fe.coarsen_grid(df_sm,  nx=2, ny=8), 'SantaMaria', weather_sm,  backfill_lags=True, drop_anomaly=True)
df_feat_sal_2x8 = fe.build_features(fe.coarsen_grid(df_sal, nx=2, ny=8), 'Salinas',    weather_sal, backfill_lags=True, drop_anomaly=True)
print('2x8 done.')

# 8x2
df_feat_sm_8x2  = fe.build_features(fe.coarsen_grid(df_sm,  nx=8, ny=2), 'SantaMaria', weather_sm,  backfill_lags=True, drop_anomaly=True)
df_feat_sal_8x2 = fe.build_features(fe.coarsen_grid(df_sal, nx=8, ny=2), 'Salinas',    weather_sal, backfill_lags=True, drop_anomaly=True)
print('8x2 done.')

GRID_FEATS = {
    '1x1': (df_feat_sm,     df_feat_sal),
    '5x5': (df_feat_sm_5x5, df_feat_sal_5x5),
    '7x7': (df_feat_sm_7x7, df_feat_sal_7x7),
    '8x8': (df_feat_sm_8x8, df_feat_sal_8x8),
    '2x8': (df_feat_sm_2x8, df_feat_sal_2x8),
    '8x2': (df_feat_sm_8x2, df_feat_sal_8x2),
}
print('\nAll grid shapes ready:')
for shape, (sm, sal) in GRID_FEATS.items():
    print(f'  {shape}: SantaMaria={len(sm):,} rows, Salinas={len(sal):,} rows')


### Cell 19 — Run cross experiment (ABCDE × all grids)

In [ ]:
cross_df = vs.run_cross_experiment(
    GRID_FEATS,
    schemes='ABCDE',
)


### Cell 20 — Summary table and heatmap

In [ ]:
vs.print_cross_summary(
    cross_df,
    grid_order   = ['1x1', '5x5', '7x7', '8x8', '2x8', '8x2'],
    scheme_order = ['A.1', 'B', 'C', 'D', 'E'],
)

vs.plot_cross_heatmap(
    cross_df,
    grid_order   = ['1x1', '5x5', '7x7', '8x8', '2x8', '8x2'],
    scheme_order = ['A.1', 'B', 'C', 'D', 'E'],
)


In [ ]:
import subprocess, sys, importlib

# Pull latest from GitHub
result = subprocess.run(['git', '-C', REPO_LOCAL, 'pull'], capture_output=True, text=True)
print(result.stdout.strip() or 'Already up to date.')

# Reload all modules
for mod in ['data_pipeline', 'feature_engineering', 'models',
            'harvest_advisor', 'visualize', 'validation_schemes']:
    sys.modules.pop(mod, None)

import data_pipeline
import feature_engineering as fe
import models as m
import harvest_advisor as ha
import visualize as viz
import validation_schemes as vs

print('✅ All modules reloaded')

In [ ]:
# Reload validation_schemes.py after editing it
import importlib
import validation_schemes as vs

vs = vs.reload_validation_schemes()

In [ ]:
grid_order   = ['1x1', '5x5', '7x7', '8x8', '2x8', '8x2']
scheme_order = ['A.1', 'B', 'C', 'D', 'E']

avg_results = vs.print_cross_average_summary(
    cross_df,
    grid_order=grid_order,
    scheme_order=scheme_order,
    test_only=True,
    display_tables=True
)

In [ ]:
avg_by_scheme = avg_results["overall_by_scheme"]
avg_by_site_scheme = avg_results["site_by_scheme"]
overall_b_compare = avg_results["overall_b_compare"]
site_b_compare = avg_results["site_b_compare"]
grid_b_compare = avg_results["grid_b_compare"]

In [ ]:
site_b_compare

## Section 4 — Cross-Site Transfer Experiment

Train on one site, evaluate on the other. Tests generalizability of spatial and weather features.

### Cell 21 — Define transfer grid shapes

In [ ]:
GRID_FEATS_TRANSFER = {
    '1x1': (df_feat_sm,     df_feat_sal),
    '7x7': (df_feat_sm_7x7, df_feat_sal_7x7),
    '8x8': (df_feat_sm_8x8, df_feat_sal_8x8),
}


### Cell 22 — Run all transfer schemes (A.1/B/C/D/E/AllWindows)

In [ ]:
full_transfer_df = vs.run_full_transfer_experiment(
    GRID_FEATS_TRANSFER,
)


### Cell 23 — Transfer summary table

In [ ]:
vs.print_full_transfer_summary(
    full_transfer_df,
    within_df=cross_df,
    grids=['1x1', '7x7', '8x8'],
)


### Cell 24 — Transfer visualisation

In [ ]:
vs.plot_full_transfer(
    full_transfer_df,
    grids=['1x1', '7x7', '8x8'],
)


##stage 2 Harvest advisor


In [ ]:

# Prerequisites (must already be in memory):
#   df_sm, df_sal           -- raw DataFrames from data_pipeline.load_site()
#   weather_sm, weather_sal -- from data_pipeline.load_weather_cached()
#   splits_sm_7x7,          -- from fe.split_data() on 7x7 features
#   splits_sal_7x7
#   model_results_sm_7x7,   -- from m.run_model_comparison() on 7x7 features
#   model_results_sal_7x7
#   GRID_FEATS              -- from Section 3 Cell 18
#


import importlib
import harvest_advisor as ha
importlib.reload(ha)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt







### CELL 1 — Field-level Method B x 7x7 x Scheme B


In [ ]:


# ── 1-A. Derive thresholds from 7x7 training split ────────────────────────────
thresholds_sm_7x7  = ha.derive_thresholds_v2(splits_sm_7x7['train'],  'SantaMaria')
thresholds_sal_7x7 = ha.derive_thresholds_v2(splits_sal_7x7['train'], 'Salinas')

# ── 1-B. Build Scheme B test windows ──────────────────────────────────────────
splits_B_sm,  _, _ = ha.build_scheme_b_splits(df_feat_sm_7x7)
splits_B_sal, _, _ = ha.build_scheme_b_splits(df_feat_sal_7x7)

print(f"SantaMaria: {len(splits_B_sm)} test windows")
print(f"Salinas   : {len(splits_B_sal)} test windows")

# ── 1-C. Run field-level Stage 2 evaluation ───────────────────────────────────
print("\n" + "=" * 65)
print("  SantaMaria — Method B x 7x7 x Scheme B")
print("=" * 65)
res_sm = ha.run_stage2_scheme_b(
    df_raw            = df_sm,
    model_results_7x7 = model_results_sm_7x7,
    weather           = weather_sm,
    thresholds        = thresholds_sm_7x7,
    site              = 'SantaMaria',
    scheme_b_splits   = splits_B_sm,
)
print(res_sm[['window','last_date','test_date','actual_days','pred_days',
              'error','correct','within_1','gr_pred',
              'velocity_raw','velocity_clipped']].to_string(index=False))
ha.print_stage2_metrics(res_sm, 'SantaMaria')

print("\n" + "=" * 65)
print("  Salinas — Method B x 7x7 x Scheme B")
print("=" * 65)
res_sal = ha.run_stage2_scheme_b(
    df_raw            = df_sal,
    model_results_7x7 = model_results_sal_7x7,
    weather           = weather_sal,
    thresholds        = thresholds_sal_7x7,
    site              = 'Salinas',
    scheme_b_splits   = splits_B_sal,
)
print(res_sal[['window','last_date','test_date','actual_days','pred_days',
               'error','correct','within_1','gr_pred',
               'velocity_raw','velocity_clipped']].to_string(index=False))
ha.print_stage2_metrics(res_sal, 'Salinas')


# ── 1-D. Visualization ────────────────────────────────────────────────────────

def plot_stage2_results(df, thresholds, site):
    """Plot pred vs actual days, error bars, and growth rate trajectory."""
    import matplotlib.gridspec as gridspec

    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

    x   = np.arange(len(df))
    t_low  = thresholds['t_low']
    t_high = thresholds['t_high']

    # Plot 1: pred vs actual days
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(x, df['actual_days'], 'o-',  color='#2d6a3f', lw=2, ms=7,
             label='actual_days (farmer)')
    ax1.plot(x, df['pred_days'],   '^--', color='#E07B39', lw=2, ms=7,
             label='pred_days (Method B)')
    for xi, row in zip(x, df.itertuples()):
        color = '#2d6a3f' if row.correct else '#c0392b'
        ax1.annotate(f"err={row.error:+d}",
                     (xi, max(row.actual_days, row.pred_days)),
                     textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=8, color=color, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"{row.test_date}\n(w{row.window})"
                          for row in df.itertuples()],
                         rotation=20, ha='right', fontsize=8)
    ax1.set_ylabel('Days since last harvest')
    ax1.set_title(f'{site} — Predicted vs Actual Harvest Interval',
                  fontsize=12, fontweight='bold')
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3); ax1.set_ylim(bottom=0)

    # Plot 2: error bars
    ax2     = fig.add_subplot(gs[1, 0])
    colours = ['#2d6a3f' if e == 0 else ('#E07B39' if abs(e)==1 else '#c0392b')
               for e in df['error']]
    ax2.bar(x, df['error'], color=colours, edgecolor='white')
    ax2.axhline(0, color='black', lw=1.5)
    ax2.axhline(df['error'].mean(), color='#E07B39', lw=2, ls='--',
                label=f"bias={df['error'].mean():+.2f}d")
    ax2.set_xticks(x)
    ax2.set_xticklabels([str(row.test_date) for row in df.itertuples()],
                         rotation=30, ha='right', fontsize=8)
    ax2.set_ylabel('error = pred - actual (days)')
    ax2.set_title('Prediction Error\ngreen=exact  orange=±1d  red=>±1d',
                  fontweight='bold')
    ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

    # Plot 3: growth rate
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(x, df['gr_pred'], '^-', color='#E07B39', lw=2, ms=7,
             label='gr_pred')
    ax3.axhline(t_high, color='#2d6a3f', ls='--', lw=1.5,
                label=f't_high={t_high:.3f}')
    ax3.axhline(t_low,  color='#c0392b', ls='--', lw=1.5,
                label=f't_low={t_low:.3f}')
    ax3.fill_between(x, t_low, t_high, alpha=0.08, color='#5B8DB8')
    for xi, row in zip(x, df.itertuples()):
        ax3.annotate(f"+{row.pred_days}d",
                     (xi, row.gr_pred),
                     textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=8, color='#E07B39', fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels([str(row.last_date) for row in df.itertuples()],
                         rotation=30, ha='right', fontsize=8)
    ax3.set_ylabel('gr_pred')
    ax3.set_title('Growth Rate & Recommended Interval', fontweight='bold')
    ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

    fig.suptitle(f'{site} — Stage 2 Method B v3 (pred vs actual days)',
                 fontsize=13, fontweight='bold')
    plt.show()

plot_stage2_results(res_sm,  thresholds_sm_7x7,  'SantaMaria')
plot_stage2_results(res_sal, thresholds_sal_7x7, 'Salinas')

### CELL 2 — Grid-level ML vs Rule-B comparison (all grid sizes x ABCDE)

In [ ]:


# Prerequisite: GRID_FEATS must be in memory (from Section 3 Cell 18)
# Format: {'1x1': (df_feat_sm, df_feat_sal),
#           '5x5': (df_feat_sm_5x5, df_feat_sal_5x5), ...}

results_df = ha.run_grid_comparison(
    grid_feats = GRID_FEATS,
    df_sm      = df_sm,
    df_sal     = df_sal,
    grid_sizes = ['1x1', '5x5', '7x7', '8x8'],   # adjust as needed
    schemes    = ['A.1', 'B', 'C', 'D', 'E'],
)

# Visualization
ha.plot_grid_comparison(
    results_df,
    grid_sizes = ['1x1', '5x5', '7x7', '8x8'],
    schemes    = ['A.1', 'B', 'C', 'D', 'E'],
)

# Full results table
print("\nFull Results Table:")
print(results_df.sort_values(['site', 'grid', 'scheme', 'method'])
      .to_string(index=False))

In [ ]:
# CELL 3 — Rule A vs Rule B direct comparison (field-level, 7x7, Scheme B)
# ───────────────────────────────────────────────────────────────────────────────
# Rule A: original Stage 2 — only gr_pred, no velocity
# Rule B: Method B         — gr_pred + velocity (acceleration)
# Same test windows, same Stage 1 model, same thresholds derivation data.

importlib.reload(ha)   # pick up latest harvest_advisor.py

comp_sm = ha.compare_rule_a_vs_b(
    df_raw            = df_sm,
    model_results_7x7 = model_results_sm_7x7,
    weather           = weather_sm,
    df_feat_train     = splits_sm_7x7['train'],
    site              = 'SantaMaria',
    scheme_b_splits   = splits_B_sm,
)
ha.plot_rule_a_vs_b(comp_sm, 'SantaMaria')

comp_sal = ha.compare_rule_a_vs_b(
    df_raw            = df_sal,
    model_results_7x7 = model_results_sal_7x7,
    weather           = weather_sal,
    df_feat_train     = splits_sal_7x7['train'],
    site              = 'Salinas',
    scheme_b_splits   = splits_B_sal,
)
ha.plot_rule_a_vs_b(comp_sal, 'Salinas')


### Save for Deployment

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DEPLOYMENT SAVE CELL
# Run at the end of your notebook after all sections are complete.
#
# Prerequisites in memory (all produced by earlier cells):
#   df_sm, df_sal                       -- Section 1 Cell 3
#   weather_sm, weather_sal             -- Section 1 Cell 4
#   df_feat_sm_7x7, df_feat_sal_7x7     -- Section 3 Cell 18
#   model_results_sm_7x7                -- Section 3 Cell 18 (7x7 model comparison)
#   model_results_sal_7x7               -- Section 3 Cell 18
#   thresholds_sm_7x7, thresholds_sal_7x7 -- Stage 2 Cell 1
#   OUTPUTS                             -- path to outputs/processed_data/
#
# Nothing is re-trained here. All artifacts are taken directly from memory.
#
# Saves to outputs/../deployment/:
#   model_sm_7x7.pkl        LightGBM, A3, 7x7  (SantaMaria best model)
#   model_sal_7x7.pkl       LightGBM+log, A4, 7x7  (Salinas best model)
#   thresholds_sm.pkl       Stage 2 Rule B thresholds (SantaMaria)
#   thresholds_sal.pkl      Stage 2 Rule B thresholds (Salinas)
#   df_raw_sm.parquet       Raw harvest data, real kg (SantaMaria)
#   df_raw_sal.parquet      Raw harvest data, real kg (Salinas)
#   weather_SantaMaria.csv  Weather data
#   weather_Salinas.csv     Weather data
#   deploy_config.json      App configuration
# ═══════════════════════════════════════════════════════════════════════════════

import os, json, shutil, pickle
import pandas as pd

DEPLOY_DIR = os.path.join(OUTPUTS, '..', 'deployment')
os.makedirs(DEPLOY_DIR, exist_ok=True)
print(f"Saving to: {os.path.abspath(DEPLOY_DIR)}\n")

# ── 1. Stage 1 models (already trained, just pickle) ─────────────────────────
print("[1/5] Saving Stage 1 models...")
with open(os.path.join(DEPLOY_DIR, 'model_sm_7x7.pkl'),  'wb') as f:
    pickle.dump(model_results_sm_7x7, f)
with open(os.path.join(DEPLOY_DIR, 'model_sal_7x7.pkl'), 'wb') as f:
    pickle.dump(model_results_sal_7x7, f)
print("  model_sm_7x7.pkl   -> LightGBM A3 7x7 (SantaMaria)")
print("  model_sal_7x7.pkl  -> LightGBM+log A4 7x7 (Salinas)")

# ── 2. Stage 2 thresholds (already derived, just pickle) ─────────────────────
print("\n[2/5] Saving Stage 2 thresholds...")
with open(os.path.join(DEPLOY_DIR, 'thresholds_sm.pkl'),  'wb') as f:
    pickle.dump(thresholds_sm_7x7, f)
with open(os.path.join(DEPLOY_DIR, 'thresholds_sal.pkl'), 'wb') as f:
    pickle.dump(thresholds_sal_7x7, f)
print("  thresholds_sm.pkl  -> Rule B (SantaMaria)")
print("  thresholds_sal.pkl -> Rule B (Salinas)")

# ── 3. Raw harvest data (real kg, used for yield maps and growth rate) ────────
print("\n[3/5] Saving raw harvest data...")
raw_cols = ['harvest_date', 'harvest_idx', 'field_x', 'field_y',
            'weight_kg', 'easting', 'northing']
df_sm[[c for c in raw_cols if c in df_sm.columns]].to_parquet(
    os.path.join(DEPLOY_DIR, 'df_raw_sm.parquet'), index=False)
df_sal[[c for c in raw_cols if c in df_sal.columns]].to_parquet(
    os.path.join(DEPLOY_DIR, 'df_raw_sal.parquet'), index=False)
print(f"  df_raw_sm.parquet  ({len(df_sm):,} rows)")
print(f"  df_raw_sal.parquet ({len(df_sal):,} rows)")

# ── 4. Weather data ───────────────────────────────────────────────────────────
print("\n[4/5] Saving weather data...")
for site, weather in [('SantaMaria', weather_sm), ('Salinas', weather_sal)]:
    src = os.path.join(OUTPUTS, f'weather_{site}.csv')
    dst = os.path.join(DEPLOY_DIR, f'weather_{site}.csv')
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"  weather_{site}.csv  (copied from cache)")
    else:
        weather.to_csv(dst)
        print(f"  weather_{site}.csv  (saved from memory)")

# ── 5. Deploy config ──────────────────────────────────────────────────────────
print("\n[5/5] Saving deploy config...")
config = {
    'SantaMaria': {
        'model_file':      'model_sm_7x7.pkl',
        'thresholds_file': 'thresholds_sm.pkl',
        'raw_data_file':   'df_raw_sm.parquet',
        'weather_file':    'weather_SantaMaria.csv',
        'coarsen_n':       7,
        'feature_config':  'A3',
        'best_model':      'LightGBM',
        'stage2_rule':     'Method B (velocity)',
    },
    'Salinas': {
        'model_file':      'model_sal_7x7.pkl',
        'thresholds_file': 'thresholds_sal.pkl',
        'raw_data_file':   'df_raw_sal.parquet',
        'weather_file':    'weather_Salinas.csv',
        'coarsen_n':       7,
        'feature_config':  'A4',
        'best_model':      'LightGBM + log(y+1)',
        'stage2_rule':     'Method B (velocity)',
    },
}
with open(os.path.join(DEPLOY_DIR, 'deploy_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print("  deploy_config.json")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  Verification")
print("=" * 55)
expected = [
    'model_sm_7x7.pkl', 'model_sal_7x7.pkl',
    'thresholds_sm.pkl', 'thresholds_sal.pkl',
    'df_raw_sm.parquet', 'df_raw_sal.parquet',
    'weather_SantaMaria.csv', 'weather_Salinas.csv',
    'deploy_config.json',
]
all_ok = True
for fname in expected:
    path = os.path.join(DEPLOY_DIR, fname)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  OK  {fname:<35} {size_mb:.1f} MB")
    else:
        print(f"  MISSING  {fname}")
        all_ok = False

print()
if all_ok:
    print("All files saved. Next steps:")
    print("  1. Download the deployment/ folder from Drive")
    print("  2. Place next to app.py")
    print("  3. streamlit run app.py")
else:
    print("Some files missing — check errors above.")

### Reload

In [ ]:
import subprocess, sys, importlib

# Pull latest from GitHub
result = subprocess.run(['git', '-C', REPO_LOCAL, 'pull'], capture_output=True, text=True)
print(result.stdout.strip() or 'Already up to date.')

# Reload all modules
for mod in ['data_pipeline', 'feature_engineering', 'models',
            'harvest_advisor', 'visualize', 'validation_schemes']:
    sys.modules.pop(mod, None)

import data_pipeline
import feature_engineering as fe
import models as m
import harvest_advisor as ha
import visualize as viz
import validation_schemes as vs

print('All modules reloaded')